# Live-view browser tool with LlamaIndex

## Overview

This notebook demonstrates Amazon Bedrock AgentCore live browser automation with LlamaIndex.

### Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Conversational                                                                   |
| Agent type          | Single                                                                           |
| Agentic Framework   | LlamaIndex                                                                       |
| LLM model           | Amazon Bedrock Claude-3 Haiku                                                   |
| Tutorial components | Using LlamaIndex to interact with browser tool live                             |
| Tutorial vertical   | Cross-vertical                                                                   |
| Example complexity  | Easy                                                                             |
| SDK used            | Amazon BedrockAgentCore Python SDK, LlamaIndex                                  |

### Tutorial Architecture

In this tutorial we will describe how to use LlamaIndex with browser tool and view the browser live.

In our example we will send natural language instructions to the LlamaIndex agent to perform tasks on the Bedrock AgentCore browser and view the browser live.

### Tutorial Key Features

- **Live DCV Viewer**: Real-time AgentCore browser session visualization with AWS DCV streaming
- **Single Browser Session**: Shared session between live viewer and Playwright automation
- **LlamaIndex Analysis**: AI-powered content analysis with Bedrock Claude
- **Screenshot Capture**: Full page and viewport screenshots
- **Simple Interface**: Single-command execution like Nova-Act/Strands
- **Universal Compatibility**: Works with any website and custom prompts

## Prerequisites

To execute this tutorial you will need:
* Python 3.12+
* AWS credentials
* Amazon Bedrock AgentCore SDK
* LlamaIndex SDK
* Playwright for browser automation

## 1. Environment Setup

In [ ]:
# Set up Python 3.12 virtual environment
!python3.12 --version
!python3.12 -m venv venv
!source venv/bin/activate && python --version

In [ ]:
# Install dependencies
!pip install --force-reinstall -U -r requirements.txt --quiet

print("✅ All dependencies installed successfully!")

## 2. Setup and Imports

In [ ]:
# Import required libraries
import asyncio
import sys
import time
import webbrowser
import json
from datetime import datetime
from pathlib import Path

# LlamaIndex imports
from llama_index.llms.bedrock_converse import BedrockConverse

# Browser and Playwright imports
from bedrock_agentcore.tools.browser_client import browser_session
from playwright.async_api import async_playwright

# Utilities
from rich.console import Console
from rich.panel import Panel
import boto3

console = Console()

# Add interactive tools to path for BrowserViewerServer
interactive_tools_path = Path().absolute().parent / "interactive_tools"
sys.path.append(str(interactive_tools_path))

try:
    from browser_viewer import BrowserViewerServer
    console.print(f"[green]✅ BrowserViewerServer imported from {interactive_tools_path}[/green]")
except ImportError as e:
    console.print(f"[red]❌ BrowserViewerServer not found: {e}[/red]")
    BrowserViewerServer = None

print("✅ All libraries imported successfully!")

## 3. Live Viewer Class Implementation

In [ ]:
class LiveViewerWithLlamaIndex:
    """
    Live browser automation with LlamaIndex - similar to Nova-Act/Strands approach
    """
    
    def __init__(self, region="us-east-1"):
        self.region = region
        self.browser_client = None
        self.viewer = None
        self.viewer_url = None
        self.results_dir = Path("live_analysis_results")
        self.results_dir.mkdir(exist_ok=True)
        
    async def run_live_analysis(self, prompt, starting_page):
        """
        Main function that runs the complete live analysis workflow
        """
        console.print(
            Panel(
                f"[bold cyan]LlamaIndex Live Browser Analysis[/bold cyan]\n\n"
                f"🎯 Task: {prompt}\n"
                f"🌐 Starting Page: {starting_page}\n"
                f"📁 Results: {self.results_dir}\n\n"
                f"[yellow]👀 Live viewer will open automatically![/yellow]",
                title="Live Analysis Session",
                border_style="blue",
            )
        )
        
        try:
            # Step 1: Initialize browser session and live viewer
            console.print("\n[cyan]🚀 Initializing browser session and live viewer...[/cyan]")
            
            # Create browser session
            self.browser_client = browser_session(self.region).__enter__()
            ws_url, headers = self.browser_client.generate_ws_headers()
            console.print(f"[green]✅ Browser session: {self.browser_client.session_id}[/green]")
            
            # Start live viewer
            if BrowserViewerServer:
                self.viewer = BrowserViewerServer(self.browser_client, port=8000)
                self.viewer_url = self.viewer.start(open_browser=False)
                console.print(f"[green]✅ Live viewer: {self.viewer_url}[/green]")
                
                # Open viewer for user to watch
                webbrowser.open(self.viewer_url)
                console.print("[yellow]👀 Watch the automation in your browser![/yellow]")
                time.sleep(3)  # Give viewer time to load
            
            # Step 2: Use Playwright to automate the SAME browser session
            console.print("\n[cyan]🤖 Setting up Playwright automation on live session...[/cyan]")
            
            # Create LlamaIndex LLM for analysis
            llm = BedrockConverse(
                model="anthropic.claude-3-haiku-20240307-v1:0",
                region_name=self.region,
                temperature=0.1,
                max_tokens=4000
            )
            
            console.print("[green]✅ LlamaIndex LLM ready[/green]")
            
            # Step 3: Execute browser automation using Playwright on the SAME session
            console.print(f"\n[cyan]🎬 Starting live browser automation...[/cyan]")
            console.print(f"[yellow]👀 Watch at: {self.viewer_url}[/yellow]")
            
            # Generate timestamp for this analysis
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            
            # Use Playwright to automate the same session the viewer is showing
            async with async_playwright() as p:
                browser = await p.chromium.connect_over_cdp(ws_url, headers=headers)
                context = browser.contexts[0] if browser.contexts else await browser.new_context()
                page = context.pages[0] if context.pages else await context.new_page()
                
                console.print(f"[green]✅ Connected to live browser session[/green]")
                
                # Navigate to starting page (user can see this!)
                console.print(f"[yellow]🌐 Navigating to: {starting_page}[/yellow]")
                await page.goto(starting_page, wait_until="domcontentloaded", timeout=30000)
                await asyncio.sleep(3)  # Let user see the navigation
                
                # Extract page content
                console.print(f"[yellow]📄 Extracting page content...[/yellow]")
                page_title = await page.title()
                page_content = await page.inner_text('body')
                
                console.print(f"[green]✅ Page loaded: {page_title}[/green]")
                console.print(f"[green]📄 Content extracted ({len(page_content)} characters)[/green]")
                
                # Take screenshots
                console.print(f"[yellow]📸 Taking screenshots...[/yellow]")
                
                # Full page screenshot
                full_screenshot = self.results_dir / f"live_analysis_{timestamp}_full.png"
                await page.screenshot(path=str(full_screenshot), full_page=True)
                
                # Viewport screenshot  
                viewport_screenshot = self.results_dir / f"live_analysis_{timestamp}_viewport.png"
                await page.screenshot(path=str(viewport_screenshot), full_page=False)
                
                console.print(f"[green]📸 Screenshots saved[/green]")
                
                # Wait for user to observe
                console.print(f"\n[yellow]👀 Check the live viewer at: {self.viewer_url}[/yellow]")
                console.print(f"[yellow]⏱️  Waiting 10 seconds for you to observe the page...[/yellow]")
                
                for i in range(10, 0, -1):
                    console.print(f"   {i} seconds...", end='\r')
                    await asyncio.sleep(1)
                
                console.print("\n")
                
                # Use LlamaIndex to analyze the content
                console.print(f"[cyan]🤖 Analyzing content with LlamaIndex...[/cyan]")
                
                analysis_prompt = f"""
                Analyze this web page content and complete the requested task:
                
                Page Title: {page_title}
                URL: {starting_page}
                Task: {prompt}
                
                Page Content:
                {page_content[:3000]}
                
                Please provide a detailed analysis focusing on the specific task requested.
                Extract the exact information requested and present it clearly.
                """
                
                response = await llm.acomplete(analysis_prompt)
                result = response.text
            
            # Step 4: Save results and provide summary
            # Save the result
            result_data = {
                "timestamp": timestamp,
                "prompt": prompt,
                "starting_page": starting_page,
                "page_title": page_title,
                "content_length": len(page_content),
                "agent_response": str(result),
                "session_id": self.browser_client.session_id,
                "viewer_url": self.viewer_url,
                "screenshots": {
                    "full_page": str(full_screenshot),
                    "viewport": str(viewport_screenshot)
                },
                "success": True
            }
            
            result_file = self.results_dir / f"live_analysis_{timestamp}.json"
            with open(result_file, 'w', encoding='utf-8') as f:
                json.dump(result_data, f, indent=2, ensure_ascii=False)
            
            # Also save as text for easy reading
            text_file = self.results_dir / f"live_analysis_{timestamp}.txt"
            with open(text_file, 'w', encoding='utf-8') as f:
                f.write(f"Live Browser Analysis Results\n")
                f.write(f"{'='*50}\n\n")
                f.write(f"Timestamp: {timestamp}\n")
                f.write(f"Task: {prompt}\n")
                f.write(f"Starting Page: {starting_page}\n")
                f.write(f"Session ID: {self.browser_client.session_id}\n")
                f.write(f"Live Viewer: {self.viewer_url}\n\n")
                f.write(f"Agent Response:\n")
                f.write(f"{'-'*30}\n")
                f.write(f"{result}\n")
            
            # Display results
            console.print(f"\n[bold green]✅ Live Analysis Complete![/bold green]")
            console.print(f"📋 Results saved: {result_file}")
            console.print(f"📄 Text summary: {text_file}")
            console.print(f"📸 Full page screenshot: {full_screenshot}")
            console.print(f"📸 Viewport screenshot: {viewport_screenshot}")
            console.print(f"👀 Live viewer: {self.viewer_url}")
            
            console.print(f"\n[bold cyan]🎯 Agent Response:[/bold cyan]")
            console.print(Panel(str(result), title="Analysis Results", border_style="green"))
            
            # Keep viewer open for observation
            console.print(f"\n[yellow]⏱️  Keeping live viewer open for 10 seconds for final observation...[/yellow]")
            await asyncio.sleep(10)
            
            return result_data
            
        except Exception as e:
            console.print(f"[red]❌ Error during live analysis: {e}[/red]")
            import traceback
            console.print(f"[dim]{traceback.format_exc()}[/dim]")
            return {
                "error": str(e),
                "success": False
            }
        
        finally:
            # Cleanup
            try:
                if self.browser_client:
                    self.browser_client.stop()
                    console.print("[green]✅ Browser session cleaned up[/green]")
            except Exception as e:
                console.print(f"[yellow]⚠️ Cleanup warning: {e}[/yellow]")

print("✅ LiveViewerWithLlamaIndex class defined!")

## 4. Initialize the System

In [ ]:
# Get AWS region
boto_session = boto3.Session()
region = boto_session.region_name or "us-east-1"

# Initialize the live analysis system
analyzer = LiveViewerWithLlamaIndex(region=region)

console.print("🚀 Live Browser Analysis System with LlamaIndex ready!")
console.print("\n📋 System capabilities:")
console.print("   ✅ Live DCV browser viewing")
console.print("   ✅ Single browser session (shared between viewer and automation)")
console.print("   ✅ Playwright automation on live session")
console.print("   ✅ Screenshot capture (full page + viewport)")
console.print("   ✅ LlamaIndex + Bedrock Claude-3 Haiku")
console.print("   ✅ Real-time content extraction and analysis")
console.print("   ✅ Universal website compatibility")
console.print(f"\n🎯 Using AWS region: {region}")
console.print("\n🎬 Ready for live browser automation!")

## 5. Usage Examples

Run these examples to see live browser automation in action!

### Example 1: Stock Analysis with Live Viewer (Tested ✅)

In [ ]:
# Analyze Apple stock with live viewer - you'll see the browser automation in real-time!
# This example has been tested and works perfectly
result = await analyzer.run_live_analysis(
    prompt="Find and extract the current stock price, market cap, and P/E ratio",
    starting_page="https://stockanalysis.com/stocks/aapl/"
)

### Example 2: News Headlines Extraction

In [ ]:
# Extract news headlines with live viewer
result = await analyzer.run_live_analysis(
    prompt="Extract the top 3 news headlines and provide a brief summary of each",
    starting_page="https://news.ycombinator.com"
)

### Example 3: Financial Data Extraction

In [ ]:
# Extract financial data with live viewer
result = await analyzer.run_live_analysis(
    prompt="Extract Tesla's current stock price, market cap, and recent performance metrics",
    starting_page="https://finance.yahoo.com/quote/TSLA"
)

### Example 4: GitHub Trending Analysis

In [ ]:
# Analyze GitHub trending repositories with live viewer
result = await analyzer.run_live_analysis(
    prompt="What are the top 5 trending repositories and what technologies are they using?",
    starting_page="https://github.com/trending"
)

## 6. Command Line Usage

You can also use the standalone script for single-command execution:

```bash
# Stock analysis (tested and working ✅)
python live_view_with_llamaindex.py --prompt "Find and extract the current stock price, market cap, and P/E ratio" --starting-page "https://stockanalysis.com/stocks/aapl/"

# News extraction  
python live_view_with_llamaindex.py --prompt "Extract the top 3 news headlines" --starting-page "https://news.ycombinator.com"


## 7. Results

Each analysis creates files in the `live_analysis_results` directory:
- `live_analysis_TIMESTAMP.json` - Complete structured results with metadata
- `live_analysis_TIMESTAMP.txt` - Human-readable text summary
- `live_analysis_TIMESTAMP_full.png` - Full page screenshot
- `live_analysis_TIMESTAMP_viewport.png` - Viewport screenshot

## What happened behind the scenes?

* The LiveViewerWithLlamaIndex class automatically manages browser session creation and live viewer integration
* You configured the system to use LlamaIndex with Bedrock Claude-3 Haiku for intelligent content analysis
* The system created a shared browser session that both the live viewer and Playwright automation can access simultaneously
* LlamaIndex took your natural language instructions and performed web automation while streaming the browser session live
* The agent navigated to the target page, extracted content, captured screenshots, and analyzed the information using AI
* All automation steps were visible in real-time through the DCV live viewer at http://localhost:8000


## What You'll Get

- **Live browser viewing** at `http://localhost:8000`
- **Real-time automation** you can watch happen
- **AI-powered analysis** of any website content
- **Screenshots** (full page + viewport)
- **Structured results** saved as JSON and text files

The system provides the same live viewing experience as Nova-Act and Strands implementations but uses LlamaIndex for intelligent content analysis!

## Test Results

After running the examples above, you'll see results like this:

**Example: Apple Stock Analysis (AAPL)**
- Current Stock Price: $255.46
- Market Cap: $3.79 Trillion  
- P/E Ratio: 38.86
- Screenshots: ✅ Captured and saved
- Live Viewer: ✅ Working at http://localhost:8000
- Analysis: ✅ Accurate extraction and analysis